# Metacatalog on sky image

Matplotlib band overlays + interactive **SkyWidget + Bokeh** catalog map + **sortable/filterable metacatalog table** for the global `metacatalog.csv` produced by `ovro_lwa_metacatalog.ipynb`.

Expected CSV columns include `bands_present` (comma-separated contributing bands), `origin_band` (band that seeded the row), canonical `RA`/`DEC`/`Peak_flux` from that seed, per-band `Peak_flux_{Blue,Green,Red}` / `RA_{band}` columns when associated, and `n_assoc_{band}` counts.

**Matplotlib:** sources on each band image when that band appears in `bands_present`. **Bokeh:** all sources after flux cut, colored by band combination.

Launch with `pixi run jupyter lab`. **Run cells in order.**


In [ ]:
from __future__ import annotations

import os
import re
from pathlib import Path

import astropy.units as u
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.visualization import ImageNormalize, PercentileInterval, AsinhStretch
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.plotting import figure
from bokeh.transform import factor_cmap
from matplotlib import pyplot as plt

from astrowidget import SkyWidget
from astrowidget.wcs import adjust_wcs_for_array_stride

# --- paths ----------------------------------------------------------------
FITS_ROOT = Path("/fast/claw")
METACATALOG_CSV = Path("/fast/claw/metacatalog/metacatalog.csv")
LST_HOUR = "01h"  # LST hour for matplotlib FITS backgrounds (catalog is LST-merged global)
BANDS = ("Full", "Red", "Green", "Blue")
COLOR_BANDS = ("Blue", "Green", "Red")

# catalog display
FLUX_PERCENTILE = None
MAX_PLOT_SOURCES = 999999

# reticle overlay
RETICLE_ARM_PIX = 14
RETICLE_GAP_FRAC = 0.35

# HiPS (same paths as source_review.ipynb)
HIPS_ROOT = Path("/lustre/pipeline/calibration/hips")
HIPS_BACKGROUND = HIPS_ROOT / "Blue_I_deep_Taper_Robust-0.75_Jan25.hips"
HIPS_HTTP_PREFIX = os.environ.get("OVRO_HIPS_HTTP_BASE", "/calibration/hips")
os.environ.setdefault("OVRO_HIPS_HTTP_BASE", HIPS_HTTP_PREFIX)
os.environ.setdefault("OVRO_HIPS_ROOT", str(HIPS_ROOT))
HIPS_BACKGROUND_PERCENTILE_LOW = 1.0
HIPS_BACKGROUND_PERCENTILE_HIGH = 99.0
BACKGROUND_CUT_MIN = None
BACKGROUND_CUT_MAX = None
BACKGROUND_OPACITY = 1.0

# SkyWidget radio overlay
OVERLAY_MAX_SIZE = 1024
OVERLAY_COLORMAP = "magma"
OVERLAY_STRETCH = "log"
OVERLAY_OPACITY = 1.0
OVERLAY_PERCENTILE_LOW = 2.0
OVERLAY_PERCENTILE_HIGH = 98.0

# Marker colors keyed by band combination (bands_present with '+' for lookup)
BAND_SIGNATURE_COLORS = {
    "Full": "#22d3ee",
    "Blue": "#60a5fa",
    "Green": "#4ade80",
    "Red": "#f87171",
    "Full+Blue": "#38bdf8",
    "Full+Green": "#2dd4bf",
    "Full+Red": "#fb7185",
    "Full+Blue+Green": "#a3e635",
    "Full+Blue+Red": "#c084fc",
    "Full+Green+Red": "#fbbf24",
    "Full+Blue+Green+Red": "#f472b6",
}
METACATALOG_REQUIRED_COLS = (
    "meta_id",
    "bands_present",
    "origin_band",
    "RA",
    "DEC",
    "Peak_flux",
)
WIDGET_FOV_DEG = 25.0
FOCUS_FOV_DEG = 8.0
BOKEH_MAP_PX = 840

# Interactive metacatalog table (Panel Tabulator)
CATALOG_TABLE_HEIGHT = 420
CATALOG_TABLE_PAGE_SIZE = 50


In [ ]:

_OVERLAY_BAND_CANON = {b.lower(): b for b in BANDS}
_FITS_OVERLAY_RE = re.compile(
    r"^I_(?P<lst>\d+h)_.*_(?P<band>Full|Red|Green|Blue)\.fits$",
    re.IGNORECASE,
)

CATALOG_COORDS = {
    "Full": ("RA", "DEC"),
    "Blue": ("RA_Blue", "DEC_Blue"),
    "Green": ("RA_Green", "DEC_Green"),
    "Red": ("RA_Red", "DEC_Red"),
}
BAND_FLUX_COL = {
    "Full": "Peak_flux",
    "Blue": "Peak_flux_Blue",
    "Green": "Peak_flux_Green",
    "Red": "Peak_flux_Red",
}


def detected_bands(row: pd.Series) -> list[str]:
    """Bands in which this metacatalog source was detected."""
    bands_present = row.get("bands_present")
    if isinstance(bands_present, str) and bands_present.strip():
        return [b.strip() for b in bands_present.split(",") if b.strip()]
    raise ValueError(
        "metacatalog row missing bands_present; regenerate metacatalog.csv with "
        "ovro_lwa_metacatalog.ipynb"
    )


def band_signature(row: pd.Series) -> str:
    return "+".join(detected_bands(row))


def signature_color(signature: str) -> str:
    return BAND_SIGNATURE_COLORS.get(signature, "#94a3b8")


def band_coord_deg(row: pd.Series, band: str) -> tuple[float, float]:
    """Sky position for one band on a metacatalog row."""
    if band == "Full":
        return float(row["RA"]), float(row["DEC"])
    ra_col, dec_col = CATALOG_COORDS[band]
    ra = float(row[ra_col])
    dec = float(row[dec_col])
    if np.isfinite(ra) and np.isfinite(dec):
        return ra, dec
    return float(row["RA"]), float(row["DEC"])


def catalog_radec_deg(row: pd.Series) -> tuple[float, float]:
    """Canonical map position (seed-band RA/DEC from sequential merge)."""
    return float(row["RA"]), float(row["DEC"])


def catalog_skycoord(row: pd.Series) -> SkyCoord:
    ra_deg, dec_deg = catalog_radec_deg(row)
    return SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame="icrs")


def catalog_flux(row: pd.Series) -> float:
    fluxes: list[float] = []
    for band in detected_bands(row):
        val = row.get(BAND_FLUX_COL[band], np.nan)
        if isinstance(val, (int, float)) and np.isfinite(val):
            fluxes.append(float(val))
    if not fluxes:
        return float("nan")
    return max(fluxes)


def prepare_mpl_band_catalog(catalog: pd.DataFrame, image_band: str) -> pd.DataFrame:
    """Sources to overlay on one matplotlib band image (no flux cut)."""
    band_lists = catalog.apply(detected_bands, axis=1)
    mask = band_lists.apply(lambda bands: image_band in bands)
    sub = catalog.loc[mask].copy()
    coords = sub.apply(lambda row: band_coord_deg(row, image_band), axis=1, result_type="expand")
    sub["cat_ra"], sub["cat_dec"] = coords[0], coords[1]
    sub = sub[np.isfinite(sub["cat_ra"]) & np.isfinite(sub["cat_dec"])]
    return sub[["cat_ra", "cat_dec"]].reset_index(drop=True)


def prepare_plot_catalog(catalog: pd.DataFrame) -> pd.DataFrame:
    plot_df = catalog.copy()
    cat_ra_dec = plot_df.apply(catalog_radec_deg, axis=1, result_type="expand")
    plot_df["cat_ra"], plot_df["cat_dec"] = cat_ra_dec[0], cat_ra_dec[1]
    plot_df = plot_df[np.isfinite(plot_df["cat_ra"]) & np.isfinite(plot_df["cat_dec"])].copy()
    plot_df["band_signature"] = plot_df.apply(band_signature, axis=1)
    plot_df["bands_present"] = plot_df["bands_present"].astype(str)
    plot_df["marker_color"] = plot_df["band_signature"].map(signature_color)
    plot_df["_flux"] = plot_df.apply(catalog_flux, axis=1)
    if FLUX_PERCENTILE is not None and len(plot_df):
        cutoff = np.nanpercentile(plot_df["_flux"], FLUX_PERCENTILE)
        plot_df = plot_df[plot_df["_flux"] >= cutoff]
    if len(plot_df) > MAX_PLOT_SOURCES:
        plot_df = plot_df.nlargest(MAX_PLOT_SOURCES, "_flux")
    return plot_df.reset_index(drop=True)


TABLE_DISPLAY_COLUMNS = (
    "meta_id",
    "ra_hms",
    "dec_dms",
    "cat_ra",
    "cat_dec",
    "origin_band",
    "bands_present",
    "band_signature",
    "Peak_flux",
    "lst_hours",
    "n_lst_contributions",
    "n_assoc_Blue",
    "n_assoc_Green",
    "n_assoc_Red",
)


def prepare_table_catalog(catalog: pd.DataFrame) -> pd.DataFrame:
    """Metacatalog rows for the interactive Tabulator view (full catalog, no flux cut)."""
    out = catalog.copy()
    coords = out.apply(catalog_radec_deg, axis=1, result_type="expand")
    out["cat_ra"], out["cat_dec"] = coords[0], coords[1]
    out = out[np.isfinite(out["cat_ra"]) & np.isfinite(out["cat_dec"])].copy()
    out["band_signature"] = out.apply(band_signature, axis=1)
    if "bands_present" in out.columns:
        out["bands_present"] = out["bands_present"].astype(str)
    sc = SkyCoord(ra=out["cat_ra"].to_numpy() * u.deg, dec=out["cat_dec"].to_numpy() * u.deg)
    out["ra_hms"] = sc.ra.to_string(unit=u.hour, sep=":", precision=1, pad=True)
    out["dec_dms"] = sc.dec.to_string(unit=u.deg, sep=":", alwayssign=True, precision=1, pad=True)
    if "Peak_flux" in out.columns:
        out["Peak_flux"] = out["Peak_flux"].round(3)
    cols = [c for c in TABLE_DISPLAY_COLUMNS if c in out.columns]
    return out[cols].reset_index(drop=True)


def discover_fits_overlays(root: Path = FITS_ROOT) -> list[dict[str, object]]:
    found: list[dict[str, object]] = []
    for path in sorted(root.rglob("I_*h_*_*.fits")):
        match = _FITS_OVERLAY_RE.match(path.name)
        if match is None:
            continue
        band = _OVERLAY_BAND_CANON.get(match.group("band").lower())
        if band is None:
            continue
        lst = match.group("lst").lower()
        found.append({"lst": lst, "band": band, "path": path, "label": f"{lst} — {band}"})
    if not found:
        msg = f"No overlay FITS under {root}/**/I_*h_*_{{Full,Red,Green,Blue}}.fits"
        raise FileNotFoundError(msg)
    return found


def load_fits_overlay(path: Path) -> tuple[np.ndarray, WCS]:
    with fits.open(path, memmap=True) as hdul:
        raw = np.squeeze(np.asarray(hdul[0].data, dtype=np.float32))
        header = hdul[0].header.copy()
        wcs_obj = WCS(header).celestial
    return np.where(np.isfinite(raw), raw, np.nan), wcs_obj


def prepare_strided_overlay(
    raw: np.ndarray,
    wcs_native: WCS,
    *,
    max_size: int = OVERLAY_MAX_SIZE,
) -> tuple[np.ndarray, WCS, int, int]:
    display_raw = np.where(np.isfinite(raw), raw, 0.0).astype(np.float32)
    n_l, n_m = display_raw.shape
    stride_l = max(1, n_l // max_size)
    stride_m = max(1, n_m // max_size)
    display_data = display_raw[::stride_l, ::stride_m]
    display_wcs = adjust_wcs_for_array_stride(wcs_native, stride_l, stride_m)
    return display_data, display_wcs, stride_l, stride_m


def resolve_fits(pattern: str, root: Path = FITS_ROOT) -> Path:
    matches = sorted(root.rglob(pattern))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected 1 match for {pattern!r} under {root}, got {len(matches)}")
    return matches[0]


def validate_metacatalog_csv(catalog: pd.DataFrame) -> None:
    missing = [col for col in METACATALOG_REQUIRED_COLS if col not in catalog.columns]
    if missing:
        msg = f"metacatalog.csv missing columns {missing}; regenerate with ovro_lwa_metacatalog.ipynb"
        raise ValueError(msg)
    if "is_full_master" in catalog.columns:
        print("NOTE: is_full_master column is legacy; expected bands_present from sequential merge")


OVERLAY_SOURCES = discover_fits_overlays()
print(f"SkyWidget overlay catalog: {len(OVERLAY_SOURCES)} FITS files")

if not METACATALOG_CSV.is_file():
    raise FileNotFoundError(f"Missing global metacatalog: {METACATALOG_CSV}")

meta = pd.read_csv(METACATALOG_CSV)
validate_metacatalog_csv(meta)
plot_meta = prepare_plot_catalog(meta)
table_meta = prepare_table_catalog(meta)  # Tabulator (full catalog)  # Bokeh map (flux cut + band-signature colors)
mpl_catalogs = {band: prepare_mpl_band_catalog(meta, band) for band in BANDS}
print(f"Global metacatalog: {len(meta)} sources")
print(f"  bands_present combinations: {meta['bands_present'].nunique()}")
print(f"Bokeh map after flux cut: {len(plot_meta)} sources")
print(f"Catalog table: {len(table_meta)} sources")
for band in BANDS:
    print(f"  Matplotlib {band}: {len(mpl_catalogs[band])} sources")

images: dict[str, dict[str, object]] = {}
for band in BANDS:
    band_path = resolve_fits(f"I_{LST_HOUR}_*_{band}.fits")
    with fits.open(band_path, memmap=True) as hdul:
        band_data = np.squeeze(np.asarray(hdul[0].data, dtype=np.float32))
        header = hdul[0].header.copy()
        band_wcs = WCS(header).celestial
    band_data = np.where(np.isfinite(band_data), band_data, np.nan)
    images[band] = {"data": band_data, "wcs": band_wcs, "path": band_path}

data = images["Full"]["data"]
wcs = images["Full"]["wcs"]

for band in BANDS:
    info = images[band]
    bw = info["wcs"]
    print(f"{band}: {info['path'].name} shape={info['data'].shape} CRVAL=({bw.wcs.crval[0]:.3f}, {bw.wcs.crval[1]:.3f})")


## 1. Matplotlib overlays


In [ ]:
%matplotlib inline


In [ ]:
plt.close("all")

from matplotlib.collections import LineCollection


def reticle_arm_deg(wcs_obj: WCS, dec_deg: float) -> tuple[float, float]:
    pix_scale_deg = float(np.mean(proj_plane_pixel_scales(wcs_obj)))
    arm_deg = RETICLE_ARM_PIX * pix_scale_deg
    cos_dec = max(float(np.cos(np.deg2rad(dec_deg))), 0.15)
    return arm_deg, arm_deg / cos_dec


def gap_crosshair_segments(ra_deg, dec_deg, *, wcs_obj: WCS, gap_frac: float):
    segments = []
    for ra, dec in zip(ra_deg, dec_deg, strict=True):
        arm_dec, arm_ra = reticle_arm_deg(wcs_obj, dec)
        gap_dec = arm_dec * gap_frac
        gap_ra = arm_ra * gap_frac
        segments.extend(
            [
                [(ra - arm_ra, dec), (ra - gap_ra, dec)],
                [(ra + gap_ra, dec), (ra + arm_ra, dec)],
                [(ra, dec - arm_dec), (ra, dec - gap_dec)],
                [(ra, dec + gap_dec), (ra, dec + arm_dec)],
            ]
        )
    return segments


def add_reticles(ax, wcs_obj: WCS, ra_deg, dec_deg) -> None:
    """White gap-crosshair markers with a dark outline."""
    if len(ra_deg) == 0:
        return
    segments = gap_crosshair_segments(ra_deg, dec_deg, wcs_obj=wcs_obj, gap_frac=RETICLE_GAP_FRAC)
    world = ax.get_transform("world")
    ax.add_collection(
        LineCollection(
            segments,
            transform=world,
            colors="black",
            linewidths=2.0,
            alpha=0.9,
            zorder=10,
            capstyle="round",
        )
    )
    ax.add_collection(
        LineCollection(
            segments,
            transform=world,
            colors="white",
            linewidths=1.2,
            alpha=1.0,
            zorder=11,
            capstyle="round",
        )
    )


def plot_band_overlay(band: str) -> plt.Figure:
    info = images[band]
    band_data, band_wcs = info["data"], info["wcs"]
    band_meta = mpl_catalogs[band]
    norm = ImageNormalize(band_data, interval=PercentileInterval(99.5), stretch=AsinhStretch())
    fig = plt.figure(figsize=(9, 9))
    ax = fig.add_subplot(1, 1, 1, projection=band_wcs)
    ax.imshow(band_data, origin="lower", cmap="inferno", norm=norm)
    add_reticles(ax, band_wcs, band_meta["cat_ra"].to_numpy(), band_meta["cat_dec"].to_numpy())
    ax.set_title(f"{band} image ({LST_HOUR}) — {len(band_meta)} sources")
    ax.set_xlabel("RA")
    ax.set_ylabel("Dec")
    ax.grid(color="white", ls=":", alpha=0.25)
    fig.tight_layout()
    return fig


for band in BANDS:
    plot_band_overlay(band)


## 2. SkyWidget + Bokeh catalog (stacked)

Sky view and catalog map render **in one cell** — sky on top, Bokeh below. Tap a marker to focus the sky view.

## 3. Interactive metacatalog table

Panel **Tabulator** table below the map: sortable columns, per-column header filters, and row selection to focus the sky view (run section 2 first).


In [ ]:
import ipywidgets as widgets
import panel as pn
from IPython.display import display

from scipy.ndimage import map_coordinates

from astrowidget.wcs import build_reproject_maps
from ovro_lwa_portal.viz.hips import compute_hips_percentile_cuts, hips_background_survey_url
from ovro_lwa_portal.viz.pipeline_qa_app import _patch_astrowidget_get_wcs

# Match source_review.ipynb: patch astrowidget WCS before creating SkyWidget.
_patch_astrowidget_get_wcs()

pn.extension("bokeh")



def _reproject_fits_for_shader(
    data: np.ndarray,
    wcs_obj: WCS,
    *,
    crval_ra: float,
    crval_dec: float,
) -> tuple[np.ndarray, WCS]:
    """Reproject a FITS C-order array onto the HiPS view tangent plane.

    Astropy ``hdul[0].data`` is ``(NAXIS2, NAXIS1)`` while astrowidget's
    ``apply_reproject_maps`` assumes numpy dim0 = WCS axis 1 (Zarr ``l``).
    Sample with ``[axis2, axis1]`` coordinates so catalog sources land on
    the same sky as HiPS after ``_push_image_frame``.
    """
    maps = build_reproject_maps(
        wcs_obj,
        data.shape,
        crval_ra=crval_ra,
        crval_dec=crval_dec,
    )
    out = map_coordinates(
        data.astype(np.float64, copy=False),
        [maps.src_m, maps.src_l],
        order=1,
        mode="constant",
        cval=np.nan,
    )
    out = out.astype(np.float32, copy=False)
    out[~maps.near_hemisphere] = np.nan
    return out, maps.wcs_out

# --- Overlay source selection (lazy load; many LST × band FITS on disk) ---
_overlay_state: dict[str, object] = {
    "data": None,
    "wcs": None,
    "lst": None,
    "band": None,
    "path": None,
}


def _apply_overlay_source(entry: dict[str, object]) -> None:
    raw, wcs_native = load_fits_overlay(entry["path"])
    display_data, display_wcs, stride_l, stride_m = prepare_strided_overlay(
        raw,
        wcs_native,
        max_size=OVERLAY_MAX_SIZE,
    )
    _overlay_state["data"] = display_data
    _overlay_state["wcs"] = display_wcs
    _overlay_state["lst"] = entry["lst"]
    _overlay_state["band"] = entry["band"]
    _overlay_state["path"] = entry["path"]
    n_l, n_m = raw.shape
    print(
        f"Overlay source: {entry['label']} ({entry['path'].name}) "
        f"display={display_data.shape} stride=({stride_l}, {stride_m}) from {n_l}×{n_m}"
    )


def _default_overlay_index() -> int:
    target_lst = str(LST_HOUR).lower()
    for idx, entry in enumerate(OVERLAY_SOURCES):
        if entry["lst"] == target_lst and entry["band"] == "Full":
            return idx
    return 0


hips_url = hips_background_survey_url(
    HIPS_BACKGROUND,
    hips_root=HIPS_ROOT,
    http_prefix=HIPS_HTTP_PREFIX,
)

sky = SkyWidget()
sky.background_survey = hips_url
sky.show_grid = True
sky.invert_horizontal_pan = True
sky.background_opacity = float(BACKGROUND_OPACITY)
sky.colormap = OVERLAY_COLORMAP
sky.stretch = OVERLAY_STRETCH
sky.opacity = float(OVERLAY_OPACITY)

if BACKGROUND_CUT_MIN is not None and BACKGROUND_CUT_MAX is not None:
    sky.background_cut_min = float(BACKGROUND_CUT_MIN)
    sky.background_cut_max = float(BACKGROUND_CUT_MAX)
else:
    try:
        cut_lo, cut_hi = compute_hips_percentile_cuts(
            HIPS_BACKGROUND,
            percentile_low=HIPS_BACKGROUND_PERCENTILE_LOW,
            percentile_high=HIPS_BACKGROUND_PERCENTILE_HIGH,
        )
        sky.background_cut_min = cut_lo
        sky.background_cut_max = cut_hi
    except (FileNotFoundError, ValueError) as exc:
        print(f"WARNING: HiPS cuts not set — {exc}")

print(f"HiPS background: {hips_url}")

center = SkyCoord(wcs.wcs.crval[0] * u.deg, wcs.wcs.crval[1] * u.deg)
sky.goto(center, fov=WIDGET_FOV_DEG * u.deg)

# View-locked overlay: reproject radio data to the current HiPS view after pan/zoom.
sky.overlay_view_lock = True

overlay_source_dropdown = widgets.Dropdown(
    options=[(entry["label"], idx) for idx, entry in enumerate(OVERLAY_SOURCES)],
    value=_default_overlay_index(),
    description="Overlay image:",
    layout=widgets.Layout(min_width="18em"),
)

overlay_toggle = widgets.Checkbox(
    value=False,
    description="Show overlay",
    indent=False,
)


def _push_fits_overlay_at_view(
    *,
    center: SkyCoord | None = None,
    fov: u.Quantity | None = None,
    update_view: bool = False,
) -> None:
    """Reproject Full-band FITS onto the active HiPS view tangent plane."""
    if center is None:
        center = sky.view_center_skycoord()
    ra_deg = float(center.icrs.ra.deg)
    dec_deg = float(center.icrs.dec.deg)
    reproj_data, reproj_wcs = _reproject_fits_for_shader(
        _overlay_state["data"],
        _overlay_state["wcs"],
        crval_ra=ra_deg,
        crval_dec=dec_deg,
    )
    sky._push_image_frame(
        reproj_data,
        reproj_wcs,
        center=center,
        fov=fov,
        update_view=update_view,
        percentile_low=OVERLAY_PERCENTILE_LOW,
        percentile_high=OVERLAY_PERCENTILE_HIGH,
    )


def set_radio_overlay(enabled: bool) -> None:
    if enabled:
        _push_fits_overlay_at_view(update_view=False)
    else:
        sky.clear_image()


def _on_view_gesture_revision(change) -> None:
    if change.get("type") != "change":
        return
    if overlay_toggle.value:
        _push_fits_overlay_at_view(update_view=False)


sky.observe(_on_view_gesture_revision, names="view_gesture_revision")
overlay_toggle.observe(lambda ch: set_radio_overlay(bool(ch["new"])), names="value")


def _on_overlay_source_change(change) -> None:
    if change.get("type") != "change":
        return
    _apply_overlay_source(OVERLAY_SOURCES[int(change["new"])])
    if overlay_toggle.value:
        _push_fits_overlay_at_view(update_view=False)


_apply_overlay_source(OVERLAY_SOURCES[_default_overlay_index()])
overlay_source_dropdown.observe(_on_overlay_source_change, names="value")


def focus_sky_widget(coord: SkyCoord, *, fov_deg: float = FOCUS_FOV_DEG) -> None:
    if overlay_toggle.value:
        _push_fits_overlay_at_view(
            center=coord,
            fov=fov_deg * u.deg,
            update_view=True,
        )
    else:
        sky.goto(coord, fov=fov_deg * u.deg)
    sky.set_crosshair(coord)

# --- NCP ZEA catalog map ----------------------------------------------------
_NCP_ZEA = WCS(naxis=2)
_NCP_ZEA.wcs.ctype = ["RA---ZEA", "DEC--ZEA"]
_NCP_ZEA.wcs.crval = [0.0, 90.0]
_NCP_ZEA.wcs.crpix = [180.0, 180.0]
_NCP_ZEA.wcs.cdelt = [-0.5, 0.5]
_NCP_ZEA.wcs.cunit = ["deg", "deg"]
_NCP_CENTER_XY = (180.0, 180.0)


def format_hms_dms(ra_deg: float, dec_deg: float) -> tuple[str, str]:
    coord = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame="icrs")
    ra_s = coord.ra.to_string(unit=u.hour, sep=":", precision=1, pad=True)
    dec_s = coord.dec.to_string(unit=u.deg, sep=":", alwayssign=True, precision=1, pad=True)
    return ra_s, dec_s


def radec_to_ncp_zea(ra_deg, dec_deg):
    ra = np.asarray(ra_deg, dtype=float)
    dec = np.asarray(dec_deg, dtype=float)
    x, y = _NCP_ZEA.all_world2pix(ra, dec, 0)
    return np.atleast_1d(np.asarray(x, dtype=float)), np.atleast_1d(np.asarray(y, dtype=float))


def ncp_zea_scalar(ra_deg: float, dec_deg: float) -> tuple[float, float]:
    x, y = radec_to_ncp_zea(ra_deg, dec_deg)
    return float(x[0]), float(y[0])


def ncp_zea_polyline(ra_deg, dec_deg):
    x, y = radec_to_ncp_zea(ra_deg, dec_deg)
    ok = np.isfinite(x) & np.isfinite(y)
    return x[ok].tolist(), y[ok].tolist()


def flux_marker_sizes_log(flux_values, *, min_size=5.0, size_ratio=3.0):
    flux = np.maximum(np.asarray(flux_values, dtype=float), np.finfo(float).tiny)
    log_f = np.log10(flux)
    log_min, log_max = float(log_f.min()), float(log_f.max())
    max_size = min_size * size_ratio
    if log_max <= log_min:
        return np.full(flux.shape, (min_size + max_size) / 2.0)
    t = (log_f - log_min) / (log_max - log_min)
    return min_size + t * (max_size - min_size)


bokeh_meta = plot_meta.copy()
hms = bokeh_meta.apply(lambda r: format_hms_dms(r["cat_ra"], r["cat_dec"]), axis=1, result_type="expand")
bokeh_meta["ra_hms"], bokeh_meta["dec_dms"] = hms[0], hms[1]

ax, ay = radec_to_ncp_zea(bokeh_meta["cat_ra"].to_numpy(), bokeh_meta["cat_dec"].to_numpy())
finite = np.isfinite(ax) & np.isfinite(ay)
bokeh_meta = bokeh_meta.loc[finite].copy().reset_index(drop=True)
ax, ay = ax[finite], ay[finite]
marker_sizes = flux_marker_sizes_log(bokeh_meta["_flux"].to_numpy())

source = ColumnDataSource(
    data=dict(
        ax=ax.tolist(),
        ay=ay.tolist(),
        cat_ra=bokeh_meta["cat_ra"].tolist(),
        cat_dec=bokeh_meta["cat_dec"].tolist(),
        ra_hms=bokeh_meta["ra_hms"].tolist(),
        dec_dms=bokeh_meta["dec_dms"].tolist(),
        peak=bokeh_meta["_flux"].tolist(),
        bands_present=bokeh_meta["bands_present"].tolist(),
        origin_band=bokeh_meta["origin_band"].astype(str).tolist(),
        band_signature=bokeh_meta["band_signature"].tolist(),
        marker_size=marker_sizes.tolist(),
    )
)

sig_factors = bokeh_meta["band_signature"].value_counts().index.tolist()
sig_palette = [signature_color(s) for s in sig_factors]
cx, cy = _NCP_CENTER_XY
eq_x, eq_y = ncp_zea_scalar(0.0, 0.0)
_map_radius = float(np.hypot(eq_x - cx, eq_y - cy))

scatter_fig = figure(
    title=f"Global metacatalog — {len(bokeh_meta)} sources (color = bands_present; tap to focus sky)",
    height=BOKEH_MAP_PX,
    width=BOKEH_MAP_PX,
    sizing_mode="fixed",
    tools="pan,wheel_zoom,box_zoom,reset,tap",
    active_tap="tap",
    match_aspect=True,
    background_fill_color="#111111",
)
_pad = 6.0
scatter_fig.x_range.start = cx - _map_radius - _pad
scatter_fig.x_range.end = cx + _map_radius + _pad
scatter_fig.y_range.start = cy - _map_radius - _pad
scatter_fig.y_range.end = cy + _map_radius + _pad

ra_circle = np.linspace(0.0, 360.0, 361)
horizon_x, horizon_y = ncp_zea_polyline(ra_circle, np.zeros(361))
meridian_xs, meridian_ys, parallel_xs, parallel_ys = [], [], [], []
for ra_hour in range(0, 24, 3):
    dec_line = np.linspace(0.0, 89.5, 120)
    xs, ys = ncp_zea_polyline(np.full(dec_line.shape, ra_hour * 15.0), dec_line)
    meridian_xs.append(xs)
    meridian_ys.append(ys)
for dec_line_val in (15, 30, 45, 60, 75):
    xs, ys = ncp_zea_polyline(np.linspace(0.0, 360.0, 361), np.full(361, dec_line_val))
    parallel_xs.append(xs)
    parallel_ys.append(ys)

scatter_fig.multi_line(
    [horizon_x] + meridian_xs + parallel_xs,
    [horizon_y] + meridian_ys + parallel_ys,
    line_color="#666666",
    line_width=0.8,
    line_alpha=0.55,
)

catalog_scatter = scatter_fig.scatter(
    "ax",
    "ay",
    source=source,
    size="marker_size",
    alpha=0.8,
    line_color="white",
    line_alpha=0.35,
    color=factor_cmap("band_signature", palette=sig_palette, factors=sig_factors),
    legend_group="band_signature",
)
scatter_fig.add_tools(
    HoverTool(
        renderers=[catalog_scatter],
        tooltips=[
            ("bands", "@bands_present"),
            ("origin", "@origin_band"),
            ("RA", "@ra_hms"),
            ("Dec", "@dec_dms"),
            ("Peak flux", "@peak{0.2f}"),
        ],
    )
)


def on_bokeh_tap(event) -> None:
    if event.x is None or event.y is None:
        return
    dist = np.hypot(np.asarray(source.data["ax"]) - event.x, np.asarray(source.data["ay"]) - event.y)
    i = int(dist.argmin())
    if dist[i] > 8.0:
        return
    focus_sky_widget(
        SkyCoord(
            ra=float(source.data["cat_ra"][i]) * u.deg,
            dec=float(source.data["cat_dec"][i]) * u.deg,
            frame="icrs",
        )
    )


scatter_fig.on_event("tap", on_bokeh_tap)

display(
    widgets.VBox(
        [widgets.HBox([overlay_toggle, overlay_source_dropdown]), sky],
        layout=widgets.Layout(width="100%", min_height="620px"),
    )
)

catalog_pane = pn.pane.Bokeh(
    scatter_fig,
    width=BOKEH_MAP_PX,
    height=BOKEH_MAP_PX,
    sizing_mode="fixed",
)
pn.Column(catalog_pane, sizing_mode="stretch_width", margin=(0, 0, 0, 0))


In [ ]:
pn.extension("tabulator")

catalog_table = pn.widgets.Tabulator(
    table_meta,
    pagination="local",
    page_size=CATALOG_TABLE_PAGE_SIZE,
    height=CATALOG_TABLE_HEIGHT,
    sizing_mode="stretch_width",
    layout="fit_data_table",
    header_filters=True,
    selectable=1,
    show_index=False,
    disabled=True,
    name="Metacatalog",
)


def _on_catalog_table_select(event) -> None:
    if not catalog_table.selection:
        return
    row = catalog_table.value.iloc[int(catalog_table.selection[0])]
    focus_sky_widget(
        SkyCoord(ra=float(row["cat_ra"]) * u.deg, dec=float(row["cat_dec"]) * u.deg, frame="icrs")
    )


catalog_table.param.watch(_on_catalog_table_select, "selection")

pn.Column(
    pn.pane.Markdown(
        f"### Metacatalog table ({len(table_meta)} sources)\n"
        "Click column headers to sort; type in the header filters to narrow rows. "
        "Select one row to slew the sky view."
    ),
    catalog_table,
    sizing_mode="stretch_width",
)
